# PubMedQA LoRA+ on Modal

Primary fine-tuning notebook for running PubMedQA LoRA+ training inside a Modal notebook.

The notebook calls `train_model`, the same function `scripts/train.py` runs. It scores the untuned base model zero-shot on PubMedQA's official 500-question test set, trains the adapter, then scores the adapter on the same questions. The report gives the difference with a paired bootstrap interval.

Assumptions:
- one strong GPU such as `A100`, `H100`, or `L40S`
- one persistent Modal Volume mounted at `PUBMEDQA_MODAL_ROOT`
- `HF_TOKEN` available in the Modal environment
- optional `WANDB_API_KEY`, `ADAPTER_HUB_REPO_ID`, and `MERGED_HUB_REPO_ID`


In [ ]:
import subprocess
import sys
from pathlib import Path

# Install the exact versions recorded in uv.lock. Pinning keeps the run
# reproducible: an unpinned `pip install -U` silently changes torch /
# transformers / peft under the notebook and invalidates the record.
PINNED = Path.cwd() / "requirements.lock.txt"
if not PINNED.exists():
    raise FileNotFoundError(
        f"{PINNED} is missing. Regenerate it with: uv export --frozen "
        "--no-hashes --no-emit-project --no-dev -o requirements.lock.txt"
    )
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PINNED)]
)


In [ ]:

import json
import os
import sys
import warnings
from pathlib import Path
from typing import Any

import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login, snapshot_download

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.pubmedqa_loraplus.config import LoRAPlusConfig

CONFIG_PATH = REPO_ROOT / "configs" / "loraplus_train.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"{CONFIG_PATH} is missing. Start the notebook from the repository root: "
        "configs/loraplus_train.json is the only source of training defaults."
    )
BASE_CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


# An empty variable (.env.example ships `BASE_MODEL_ID=`) means "not set", so it
# falls back to the config value instead of overriding it with an empty string.
def env_str(name: str, default: Any) -> Any:
    return os.getenv(name) or default


def env_int(name: str, default: int) -> int:
    return int(env_str(name, default))


def env_float(name: str, default: float) -> float:
    return float(env_str(name, default))


if os.getenv("HF_TOKEN"):
    login(
        token=os.environ["HF_TOKEN"],
        add_to_git_credential=False,
    )

VOLUME_ROOT = Path(env_str("PUBMEDQA_MODAL_ROOT", BASE_CONFIG["modal_mount_root"]))
RUN_NAME = env_str("RUN_NAME", BASE_CONFIG["run_name"])
BASE_MODEL_DIR = VOLUME_ROOT / "base_model"
DATASET_DIR = VOLUME_ROOT / "dataset"
RUN_DIR = VOLUME_ROOT / "runs" / RUN_NAME
REPORT_DIR = VOLUME_ROOT / "reports"
for path in [VOLUME_ROOT, BASE_MODEL_DIR, DATASET_DIR.parent, RUN_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = env_str("BASE_MODEL_ID", BASE_CONFIG["base_model_id"])
# The pinned SHA is a commit of the configured model, so it cannot apply to an override.
PINNED_REVISION = (
    BASE_CONFIG["base_model_revision"]
    if BASE_MODEL_ID == BASE_CONFIG["base_model_id"]
    else None
)

CONFIG = {
    **BASE_CONFIG,
    "run_name": RUN_NAME,
    "wandb_run_name": RUN_NAME,
    "base_model_id": BASE_MODEL_ID,
    "base_model_revision": env_str("BASE_MODEL_REVISION", PINNED_REVISION),
    "base_model_path": str(BASE_MODEL_DIR),
    "dataset_disk_path": str(DATASET_DIR),
    "output_root": str(RUN_DIR),
    "reports_root": str(REPORT_DIR),
    "modal_mount_root": str(VOLUME_ROOT),
    "model_dtype": env_str("MODEL_DTYPE", BASE_CONFIG["model_dtype"]),
    "train_batch_size": env_int("TRAIN_BATCH_SIZE", BASE_CONFIG["train_batch_size"]),
    "eval_batch_size": env_int("EVAL_BATCH_SIZE", BASE_CONFIG["eval_batch_size"]),
    "gradient_accumulation_steps": env_int(
        "GRADIENT_ACCUMULATION_STEPS", BASE_CONFIG["gradient_accumulation_steps"]
    ),
    "learning_rate": env_float("LEARNING_RATE", BASE_CONFIG["learning_rate"]),
    "loraplus_lr_ratio": env_float("LORAPLUS_LR_RATIO", BASE_CONFIG["loraplus_lr_ratio"]),
    "loraplus_lr_embedding": env_float(
        "LORAPLUS_LR_EMBEDDING", BASE_CONFIG["loraplus_lr_embedding"]
    ),
    "num_epochs": env_int("NUM_EPOCHS", BASE_CONFIG["num_epochs"]),
    "logging_steps": env_int("LOGGING_STEPS", BASE_CONFIG["logging_steps"]),
    "save_steps": env_int("SAVE_STEPS", BASE_CONFIG["save_steps"]),
    "eval_steps": env_int("EVAL_STEPS", BASE_CONFIG["eval_steps"]),
    "max_length": env_int("MAX_LENGTH", BASE_CONFIG["max_length"]),
    "mmlu_eval_samples": env_int("MMLU_EVAL_SAMPLES", BASE_CONFIG["mmlu_eval_samples"]),
    "wandb_project": env_str("WANDB_PROJECT", BASE_CONFIG["wandb_project"]),
    "adapter_hub_repo_id": env_str(
        "ADAPTER_HUB_REPO_ID", BASE_CONFIG.get("adapter_hub_repo_id")
    ),
    "merged_hub_repo_id": env_str("MERGED_HUB_REPO_ID", BASE_CONFIG.get("merged_hub_repo_id")),
}

os.environ.setdefault("WANDB_PROJECT", CONFIG["wandb_project"])
CONFIG_OBJ = LoRAPlusConfig(**CONFIG)

print(
    {
        "repo_root": str(REPO_ROOT),
        "modal_root": str(VOLUME_ROOT),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "cpu"
        ),
        "device_count": torch.cuda.device_count(),
        "base_model_id": CONFIG["base_model_id"],
        "base_model_revision": CONFIG["base_model_revision"],
        "run_name": RUN_NAME,
    }
)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is intended to run on a Modal GPU instance.")

In [ ]:

if DATASET_DIR.exists() and any(DATASET_DIR.iterdir()):
    print(f"Using cached dataset at {DATASET_DIR}")
else:
    dataset_source = load_dataset(
        CONFIG["dataset_id"],
        CONFIG["dataset_config"],
        revision=CONFIG.get("dataset_revision"),
    )
    dataset_source.save_to_disk(str(DATASET_DIR))
    print(f"Saved dataset to {DATASET_DIR}")


def base_model_is_complete(model_dir: Path) -> bool:
    index_path = model_dir / "model.safetensors.index.json"
    if index_path.exists():
        index_payload = json.loads(index_path.read_text(encoding="utf-8"))
        weight_map = index_payload.get("weight_map", {})
        required_files = sorted(set(weight_map.values()))
        missing_files = [name for name in required_files if not (model_dir / name).exists()]
        if missing_files:
            print("Base model cache is incomplete; missing shards:", missing_files)
            return False
        return True

    fallback_files = [
        model_dir / "consolidated.safetensors",
        model_dir / "model.safetensors",
        model_dir / "pytorch_model.bin",
    ]
    return any(path.exists() for path in fallback_files)


# A cached snapshot is reused as is. Delete base_model/ on the volume to
# download it again at the pinned revision.
if base_model_is_complete(BASE_MODEL_DIR):
    print(f"Using cached base model at {BASE_MODEL_DIR}")
else:
    snapshot_download(
        repo_id=CONFIG["base_model_id"],
        local_dir=str(BASE_MODEL_DIR),
        revision=CONFIG.get("base_model_revision"),
        token=(
            os.getenv(CONFIG.get("hub_token_env", "HF_TOKEN"))
            or None
        ),
    )
    print(f"Saved base model to {BASE_MODEL_DIR}")

from src.pubmedqa_loraplus.data import load_pubmedqa

# The split train_model uses. `test` is PubMedQA's official 500-question set.
# `validation` (checkpoint selection) and `train` come from the other 500.
dataset = load_pubmedqa(CONFIG_OBJ)
print(dataset)
print(dataset["train"][0])


In [ ]:
from src.pubmedqa_loraplus.workflow import export_artifacts, train_model


def flatten_numeric_metrics(prefix: str, payload: dict[str, Any]) -> dict[str, float]:
    metrics: dict[str, float] = {}
    for key, value in payload.items():
        metric_key = f"{prefix}/{key}" if prefix else str(key)
        if isinstance(value, dict):
            metrics.update(flatten_numeric_metrics(metric_key, value))
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            metrics[metric_key] = float(value)
    return metrics


HEADLINE_KEYS = (
    "accuracy",
    "macro_f1",
    "confidence_intervals",
    "majority_class_rate",
    "max_prediction_share",
    "collapsed",
    "prediction_counts",
    "warnings",
)


def headline(evaluation: dict[str, Any]) -> dict[str, Any]:
    pubmedqa = evaluation["pubmedqa"]
    return {
        **{key: pubmedqa.get(key) for key in HEADLINE_KEYS},
        "mmlu_medical_accuracy": evaluation["mmlu_medical"]["overall_accuracy"],
    }


# Writes reports/<RUN_NAME>.json, the per-example predictions for the adapter
# and the baseline, and the adapter itself, all under the Modal volume.
report = train_model(CONFIG_OBJ)
evaluation = report["evaluation"]
if getattr(wandb, "run", None) is not None:
    wandb.log(flatten_numeric_metrics("evaluation", evaluation))

baseline = evaluation.get("baseline")
{
    "report": str(CONFIG_OBJ.report_path),
    "test": headline(evaluation),
    "baseline": headline(baseline) if baseline else None,
    "delta_vs_baseline": evaluation.get("delta_vs_baseline"),
}


In [ ]:
export_payload = export_artifacts(CONFIG_OBJ)
export_payload["hub"] = {
    "adapter_hub_repo_id": CONFIG.get("adapter_hub_repo_id"),
    "merged_hub_repo_id": CONFIG.get("merged_hub_repo_id"),
}
(CONFIG_OBJ.reports_root / f"{RUN_NAME}_exports.json").write_text(
    json.dumps(export_payload, indent=2),
    encoding="utf-8",
)
export_payload


In [ ]:
# Optional: publish the artifacts. Skipped unless a repo id is configured.
# Uses the same code path as scripts/push_to_hub.py. The Hub repo is created
# private unless `hub_private` is explicitly False in the config.
from src.pubmedqa_loraplus.hub import push_artifact_to_hub

for artifact in ("adapter", "merged"):
    if artifact == "adapter":
        repo_id = CONFIG_OBJ.adapter_hub_repo_id
    else:
        repo_id = CONFIG_OBJ.merged_hub_repo_id
    if not repo_id:
        print(f"Skipping {artifact}: no Hub repo id configured")
        continue
    print(push_artifact_to_hub(CONFIG_OBJ, artifact))
